# Joint masked encoder–decoder for bat hippocampus

A **single transformer with one shared latent space** that unifies the neural
*encoder* (`preprocess-data-v0j-share.ipynb`, behaviour → spikes) and the position
*decoder* (`decode-position-comparison.ipynb`, spikes → position). Following the
NEDS / POYO+ / OmniMouse lineage, encoding, decoding, infilling and forecasting are
all the *same* model under different **masking** schemes — no task-specific heads,
one set of weights.

**Design choices** (some overruling the auto-generated plan, per the brief):

* **Behaviour = 3-D position only** `(x, y, z)`.
* **Per-timestep, per-session tokenization** (NEDS-style), *not* POYO+ per-neuron
  tokens. The data are small and the model must be ~1 M params on 1 GPU; per-session
  linear read-in/read-out matches the existing `Bat_Transformer` / `Bat_Transformer_Decoder`
  and still yields interrogable per-neuron embeddings (read-in weight columns).
* **Two modality streams** (neural, behaviour), concatenated into a `2T` sequence;
  full bidirectional attention so each modality conditions on the other.
* **Timestamp-aware RoPE** (reuses the package `MultiheadAttentionRoPE`) so a neural
  and a behaviour token at the *same* time bin share rotary phase.
* **`ELU+1` (`ELU1`)** rate nonlinearity for the neural read-out, as in the existing
  encoding models; **Poisson NLL** for spikes, **MSE** for z-scored position.
* Transformer depth swept over **1, 2, 4, 6, 8, 10, 12 layers**.
* Generic metrics: encoding **corr / FDE** (not bits-per-spike); decoding
  **2-D / 3-D position error + discrete Fréchet** (not R²).

Masked tokens are replaced by a learned per-modality mask token and kept in the
sequence (MAE/NEDS style), so every task is a choice of which tokens are targets.

## 1 · Imports, device, seed

In [ ]:
import os, sys, math, copy, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
import tqdm

# digitalbat on path (notebooks/ -> repo root)
sys.path.insert(0, os.path.abspath('..'))

from digitalbat.datasets.data import bat_data_loader
from digitalbat.datasets.defaults import DEFAULT_SESSION_NAMES, make_data_config
from digitalbat.models.layers import MultiheadAttentionRoPE, ELU1
from digitalbat.decoding import discrete_frechet
from digitalbat.utility.utils import (
    corr, poisson_fde_population, round_robin_session_batches,
    parse_bat, bat_palette, set_random_state,
    wilcoxon_safe, stars_for_p, add_significance_bracket,
)

sns.set_context('notebook')
SEED = 0
set_random_state(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## 2 · Data

Standard `bat_data_loader` with the canonical config (`behavior_dim == 3`,
`bin_width = 0.10 s`, flight-only, raw spike counts). Batches are
`DataSample(behavior (B,T,3 z-scored), spike (B,T,K counts), mask (B,T),
flight_path_index, flight_type_index)`.

In [ ]:
# Datasets 1-3 only (no Krishna sessions): rooms are indoor lab vs Dataset 2.
data_config = make_data_config(session_names=DEFAULT_SESSION_NAMES)
dataloaders, behavior_dim, unit_num_dict, meta_info_dict = bat_data_loader(**data_config)

assert behavior_dim == 3, f'expected 3-D position, got behavior_dim={behavior_dim}'
sessions = list(unit_num_dict.keys())
print(f'{len(sessions)} sessions, behavior_dim={behavior_dim}, '
      f'{sum(unit_num_dict.values())} neurons total')
print('neurons/session:', {s: unit_num_dict[s] for s in sessions[:3]}, '...')

### 2.1 · Neuron metadata (for latent interrogation)

Build a per-neuron table (bat, session, probe, putative cell type, depth/lateral
position) so the learned neuron embeddings can be coloured by anatomy later.

In [ ]:
def build_neuron_meta(unit_num_dict, meta_info_dict):
    rows = []
    for s, k in unit_num_dict.items():
        ptype = np.asarray(meta_info_dict['pType'][s]).ravel()
        probe = np.asarray(meta_info_dict['probe_id'][s]).ravel()
        pos = np.asarray(meta_info_dict['neuron_position'][s])
        for j in range(k):
            rows.append(dict(
                session=s, bat=parse_bat(s), neuron_idx=j,
                pType=ptype[j] if j < len(ptype) else np.nan,
                probe_id=int(probe[j]) if j < len(probe) else -1,
                pos_lateral=float(pos[j, 0]) if pos.ndim == 2 and j < len(pos) else np.nan,
                pos_depth=float(pos[j, 1]) if pos.ndim == 2 and pos.shape[1] > 1 and j < len(pos) else np.nan,
            ))
    return pd.DataFrame(rows)

neuron_meta = build_neuron_meta(unit_num_dict, meta_info_dict)
print(neuron_meta.shape)
neuron_meta.head()

### 2.2 · Room map (different arenas have different geometry)

`(x, y, z)` is **not** a common frame across sessions: with Datasets 1-3, Dataset 2 is a
physically different arena from the Datasets 1/3 lab (bounds live in `meta_info_dict['r_lim']`),
so here the rooms are **`lab` vs `dataset_2`** (the `krishna_fs` branch only fires if Krishna
sessions are added). Per-session z-scoring (`behavior_mean`/`behavior_std`) already removes the
*scale* difference, but the same z-scored coordinate still means a different physical place per
arena, and place fields remap. Following the project convention (`neurons_across_envs.ipynb`,
`Bat_Transformer`'s `session_to_room`), we group sessions into **rooms** and make the
behaviour read-in/read-out **per-room** (§3) — the model's only across-room-shared maps;
neuron maps stay per-session and already carry room identity. `r_lim` is used for plot
limits only, never as model input. Set `session_to_room=None` for a single shared frame.

In [ ]:
def session_room(s):
    """3-config rooms (matches neurons_across_envs.ipynb session_room3):
    Krishna field station / Dataset 2 / indoor lab (Datasets 1 & 3 + Krishna lab)."""
    if '_fs' in s:
        return 'krishna_fs'
    if s.startswith('Dataset_2_'):
        return 'dataset_2'
    return 'lab'

session_to_room = {s: session_room(s) for s in sessions}
print({r: sum(v == r for v in session_to_room.values())
       for r in sorted(set(session_to_room.values()))})

## 3 · The joint model

One transformer over a `2T` token sequence: the first `T` tokens are **neural**
(per-session linear read-in of spike counts), the next `T` are **behaviour**
(**per-room** linear read-in of 3-D position; §2.2). Masked tokens become a learned
per-modality mask token. Per-session linear read-outs map the neural latent back to a
Poisson rate (`ELU+1`); a **per-room** read-out maps the behaviour latent back to position.

`build_rotary_emb_from_timestamps` gives both modalities matching rotary phase per
time bin (the packaged `build_rotary_emb` keys off sequence index, which would
mis-phase the concatenated sequence).

The schematic below summarises the data flow; the masking panel shows how a single
set of weights serves every task.

In [ ]:
# Architecture schematic (also saved to ./out/joint_architecture.png)
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

def _draw_architecture(ax):
    C = dict(neural='#f4cccc', beh='#cfe2f3', shared='#d9ead3', core='#fff2cc',
             out='#ead1dc', note='#f3f3f3')
    def box(x, y, w, h, text, fc, fs=9):
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.02,rounding_size=0.06',
                                    fc=fc, ec='#444', lw=1.2, zorder=2))
        ax.text(x + w / 2, y + h / 2, text, ha='center', va='center', fontsize=fs, zorder=3)
    def arrow(p0, p1):
        ax.add_patch(FancyArrowPatch(p0, p1, arrowstyle='-|>', mutation_scale=14,
                                     lw=1.6, color='#333', zorder=1))
    ax.set_xlim(0, 100); ax.set_ylim(0, 100); ax.axis('off')
    box(1, 70, 15, 12, 'Spikes\n(B, T, K)\nraw counts', C['neural'])
    box(20, 70, 16, 12, 'Per-session\nLinear read-in\n(√counts)', C['neural'])
    box(40, 70, 15, 12, '+ modality emb\n+ mask token\n(if target)', C['note'], 8.5)
    box(1, 18, 15, 12, 'Position\n(B, T, 3)\nz-scored xyz', C['beh'])
    box(20, 18, 16, 12, 'Per-room Linear\nread-in', C['beh'])
    box(40, 18, 15, 12, '+ modality emb\n+ mask token\n(if target)', C['note'], 8.5)
    box(59, 44, 17, 14, 'Concat -> (B, 2T, D)\n[ neural_1..T |\n  behaviour_1..T ]\ntimestamp RoPE', C['shared'], 8.5)
    box(80, 44, 18, 14, 'Joint transformer\nx L  (depth swept)\nfull attention,\npre-norm + FFN', C['core'])
    box(80, 70, 18, 12, 'Per-session read-out\n-> ELU+1 -> Poisson\nrate (B, T, K)', C['out'], 8.5)
    box(80, 18, 18, 12, 'Per-room read-out\n-> position\n(B, T, 3)', C['out'], 8.5)
    for p0, p1 in [((16, 76), (20, 76)), ((36, 76), (40, 76)), ((16, 24), (20, 24)),
                   ((36, 24), (40, 24)), ((55, 74), (62, 57)), ((55, 26), (62, 45)),
                   ((76, 51), (80, 51)), ((89, 58), (89, 70)), ((89, 44), (89, 30))]:
        arrow(p0, p1)
    ax.text(89, 85.5, 'Poisson NLL  (masked neural targets)', ha='center', va='bottom',
            fontsize=8, color='#7a3b52')
    ax.text(89, 16.5, 'lam x MSE  (masked behaviour targets)', ha='center', va='top',
            fontsize=8, color='#3b5a7a')
    ax.text(2, 60, 'One model, task = choice of mask  (target = predicted):',
            fontsize=9, fontweight='bold')
    rows = [('encode', 'neural target | behaviour seen', 'spikes <- behaviour'),
            ('decode', 'neural seen | behaviour target', 'position <- spikes  (headline)'),
            ('within', 'random 50% each, independent', 'within-modality infill'),
            ('cross', 'random 50%, same tokens', 'cross-modal infill'),
            ('forecast', '2nd half of both = target', 'predict the future')]
    for i, (name, mask, task) in enumerate(rows):
        y = 55 - i * 3.0
        ax.text(3, y, name, fontsize=8.2, fontweight='bold', family='monospace', va='center')
        ax.text(15, y, mask, fontsize=8, va='center', color='#333')
        ax.text(40, y, '->  ' + task, fontsize=8, va='center', color='#555')

fig, ax = plt.subplots(figsize=(13.5, 7.4))
_draw_architecture(ax)
ax.set_title('Joint masked encoder-decoder (NEDS-style): per-timestep, per-session tokens; '
             'shared latent;\nencode / decode / infill / forecast via masking', fontsize=11)
os.makedirs('./out', exist_ok=True)
fig.savefig('./out/joint_architecture.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
def build_rotary_emb_from_timestamps(timestamps, dim_head, device):
    """RoPE sin/cos from an explicit per-token timestamp vector (L,) -> (L, dim_head//2)."""
    half = dim_head // 2
    freq = 1.0 / (10000 ** (torch.arange(half, device=device, dtype=torch.float32) / half))
    angles = torch.einsum('t,f->tf', timestamps.to(device=device, dtype=torch.float32), freq)
    return angles.sin(), angles.cos()


class JointBlock(nn.Module):
    """Timestamp-RoPE multi-head attention + FFN, pre-norm residual (mirrors
    TransformerEncoderRoPE but takes externally-built sin/cos)."""
    def __init__(self, hidden_dim, num_heads, ff_dim, dropout, nonlin=nn.GELU()):
        super().__init__()
        self.attn = MultiheadAttentionRoPE(hidden_dim, num_heads)
        self.ff = nn.Sequential(nn.Linear(hidden_dim, ff_dim), nonlin,
                                nn.Linear(ff_dim, hidden_dim))
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, sin, cos, attn_mask, return_attn_weights=False):
        if return_attn_weights:
            a, aw = self.attn(x, x, x, sin=sin, cos=cos, attn_mask=attn_mask,
                              return_attn_weights=True)
        else:
            a, aw = self.attn(x, x, x, sin=sin, cos=cos, attn_mask=attn_mask), None
        x = self.norm1(x + self.dropout(a))
        x = self.norm2(x + self.dropout(self.ff(x)))
        return (x, aw) if return_attn_weights else x


class JointMaskedTransformer(nn.Module):
    """Joint masked encoder-decoder over neural + 3-D-position modalities.

    Per-session spike read-in/read-out (read-in weight columns are per-neuron
    embeddings); shared behaviour read-in/read-out. Masked tokens -> learned
    per-modality mask token, kept in the sequence (MAE/NEDS). Returns
    (spike_rate (B,T,K), beh_pred (B,T,3), latent (B,2T,D))."""
    def __init__(self, unit_num_dict, behavior_dim=3, latent_dim=64, num_heads=4,
                 ff_dim=64, num_layers=2, dropout=0.2, input_transform='sqrt',
                 session_to_room=None):
        super().__init__()
        self.latent_dim, self.num_heads = latent_dim, num_heads
        self.input_transform = input_transform
        self.session_to_room = session_to_room
        self.spike_readin = nn.ModuleDict({s: nn.Linear(k, latent_dim)
                                           for s, k in unit_num_dict.items()})
        self.spike_readout = nn.ModuleDict({s: nn.Linear(latent_dim, k)
                                            for s, k in unit_num_dict.items()})
        # Behaviour read-in/read-out: per-room (env-keyed) when session_to_room is
        # given, else shared. Rooms differ in geometry, so the position<->latent maps
        # are room-specific; neuron maps stay per-session (they already carry room id).
        if session_to_room is None:
            self.beh_readin = nn.Linear(behavior_dim, latent_dim)
            self.beh_readout = nn.Linear(latent_dim, behavior_dim)
        else:
            rooms = sorted(set(session_to_room.values()))
            self.beh_readin = nn.ModuleDict({r: nn.Linear(behavior_dim, latent_dim) for r in rooms})
            self.beh_readout = nn.ModuleDict({r: nn.Linear(latent_dim, behavior_dim) for r in rooms})
        self.modality_embed = nn.Embedding(2, latent_dim)         # 0 neural, 1 behaviour
        self.mask_token = nn.Parameter(torch.zeros(2, latent_dim))
        nn.init.normal_(self.mask_token, std=0.02)
        self.readin_dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([JointBlock(latent_dim, num_heads, ff_dim, dropout)
                                     for _ in range(num_layers)])
        self.spike_act = ELU1()

    def forward(self, session_name, spike, behavior, valid_mask,
                neural_target, beh_target, return_attn_weights=False):
        B, T, _ = behavior.shape
        dev = behavior.device
        sp = spike.float()
        if self.input_transform == 'sqrt':
            sp = torch.sqrt(torch.clamp(sp, min=0.0))
        beh_readin = (self.beh_readin if self.session_to_room is None
                      else self.beh_readin[self.session_to_room[session_name]])
        beh_readout = (self.beh_readout if self.session_to_room is None
                       else self.beh_readout[self.session_to_room[session_name]])
        z_n = self.readin_dropout(self.spike_readin[session_name](sp))
        z_b = self.readin_dropout(beh_readin(behavior))
        z_n = torch.where(neural_target.bool().unsqueeze(-1), self.mask_token[0], z_n)
        z_b = torch.where(beh_target.bool().unsqueeze(-1), self.mask_token[1], z_b)
        z_n = z_n + self.modality_embed(torch.zeros(1, dtype=torch.long, device=dev))
        z_b = z_b + self.modality_embed(torch.ones(1, dtype=torch.long, device=dev))
        x = torch.cat([z_n, z_b], dim=1)                          # (B, 2T, D)

        ts = torch.cat([torch.arange(T, device=dev), torch.arange(T, device=dev)])
        sin, cos = build_rotary_emb_from_timestamps(ts, self.latent_dim // self.num_heads, dev)
        vm = torch.cat([valid_mask, valid_mask], dim=1).float()
        attn_mask = torch.einsum('bi,bj->bij', vm, vm).bool()[:, None]

        attn_all = []
        for blk in self.blocks:
            if return_attn_weights:
                x, aw = blk(x, sin, cos, attn_mask, return_attn_weights=True)
                attn_all.append(aw)
            else:
                x = blk(x, sin, cos, attn_mask)
        h_n, h_b = x[:, :T], x[:, T:]
        rate = self.spike_act(self.spike_readout[session_name](h_n))
        beh = beh_readout(h_b)
        if return_attn_weights:
            return rate, beh, x, attn_all
        return rate, beh, x


for nl in (1, 4, 12):
    _m = JointMaskedTransformer(unit_num_dict, num_layers=nl, session_to_room=session_to_room)
    print(f'num_layers={nl}: {sum(p.numel() for p in _m.parameters())/1e6:.3f} M params')
del _m

## 4 · Masking schemes

Six tasks (per-group behaviour masking collapses to all-or-nothing with a single
3-D-position group, so it is folded into `encode`). `*_target == 1` marks tokens
masked on input (to be predicted); padded timesteps are never targets.

| scheme | neural | behaviour | task |
|---|---|---|---|
| `encode`   | masked (target) | visible | spikes ← behaviour |
| `decode`   | visible | masked (target) | **position ← spikes** (headline) |
| `within`   | random 50 % | random 50 % (independent) | within-modality infilling |
| `cross`    | random 50 % | random 50 % (same tokens) | cross-modal infilling |
| `forecast` | 2nd half target | 2nd half target | predict the future |

In [ ]:
SCHEMES = ['encode', 'decode', 'within', 'cross', 'forecast']

def sample_mask(valid_mask, scheme, frac=0.5):
    """Return (neural_target, beh_target) (B,T) float; 1 = masked target."""
    B, T = valid_mask.shape
    v = valid_mask.bool()
    z = torch.zeros_like(valid_mask)
    if scheme == 'encode':
        return valid_mask.clone(), z.clone()
    if scheme == 'decode':
        return z.clone(), valid_mask.clone()
    if scheme == 'within':
        nm = (torch.rand_like(valid_mask) < frac) & v
        bm = (torch.rand_like(valid_mask) < frac) & v
        return nm.float(), bm.float()
    if scheme == 'cross':
        m = (torch.rand_like(valid_mask) < frac) & v
        return m.float(), m.float()
    if scheme == 'forecast':
        tgt = torch.zeros_like(valid_mask)
        for b in range(B):
            idx = torch.where(v[b])[0]
            if len(idx) >= 2:
                tgt[b, idx[len(idx) // 2]:] = valid_mask[b, idx[len(idx) // 2]:]
        return tgt.clone(), tgt.clone()
    raise ValueError(scheme)


def joint_loss(rate, beh, spike, behavior, valid_mask, neural_target, beh_target,
               lam, pois_fn):
    """Poisson NLL on masked+valid neural targets + lam * MSE on behaviour targets."""
    nsel = valid_mask.bool() & neural_target.bool()
    bsel = valid_mask.bool() & beh_target.bool()
    loss = rate.new_zeros(())
    if nsel.any():
        loss = loss + pois_fn(rate[nsel], spike.float()[nsel]).mean()
    if bsel.any():
        loss = loss + lam * ((beh[bsel] - behavior[bsel]) ** 2).mean()
    return loss

## 5 · Trainer

Round-robin over sessions (`round_robin_session_batches`), one masking scheme
sampled per batch. Both validation metrics are tracked every epoch: the decode
median 3-D position error (metres, ↓) and the encode median FDE (↑).

**Model selection — joint criterion weighting FDE equally.** Rather than selecting
on decoding alone, we keep a **Pareto front** over `(−decode-error, +FDE)` during
training (a CPU checkpoint is stored only when an epoch is non-dominated, so memory
is tiny), then pick the front member maximising a **reference-scale** score

$$ J = w\,\frac{\mathrm{FDE}}{\mathrm{fde\_ref}} - (1-w)\,\frac{\mathrm{err}}{\mathrm{err\_ref}},\qquad w = 0.5. $$

Decode error is ~1 m but FDE is ~0.01, so dividing each by a typical magnitude
(`err_ref = 1.0 m`, `fde_ref = 0.01`) puts them on a common raw scale: a 0.01 gain
in FDE then counts as much as a 1 m gain in error — i.e. **FDE is up-weighted ~100×**
(the ratio `err_ref / fde_ref`). Tune `err_ref`/`fde_ref` to change the trade-off, or
`fde_weight` to tilt it further; `select='decode'` recovers decode-only selection.
(A std-based z-score would instead equalise by each metric's *spread*, giving a
data-dependent factor rather than a fixed, interpretable ~100×.)

`lam` weights the behaviour (MSE) loss vs the spike (Poisson) loss **during
training**. It is a fixed hyperparameter here (`lam=1.0`, *not* tuned); with z-scored
position the MSE term is ~O(1), comparable to the Poisson term. Selection is now
scale-free regardless of `lam`, so `lam` only sets the *gradient* balance — §6.1
sweeps it explicitly.

In [ ]:
@torch.no_grad()
def _decode_median_err_m(model, loaders, meta, device):
    """Mean over sessions of median 3-D decode error (metres) on a split."""
    model.eval()
    per_session = []
    for s, sd in loaders.items():
        std = np.asarray(meta['behavior_std'][s]).squeeze()[:3]
        errs = []
        for b in sd:
            m = b.mask.to(device)
            bt = b.mask.clone().to(device)            # decode: all behaviour masked
            nt = torch.zeros_like(bt)
            _, beh_pred, _ = model(s, b.spike.to(device), b.behavior.to(device), m, nt, bt)
            yp = beh_pred.cpu().numpy()
            yt = b.behavior[:, :, :3].numpy()
            mk = b.mask.numpy().astype(bool)
            resid = (yt - yp)[mk] * std[None, :]
            if resid.size:
                errs.append(np.sqrt((resid ** 2).sum(axis=1)))
        if errs:
            per_session.append(float(np.median(np.concatenate(errs))))
    return float(np.mean(per_session)) if per_session else np.inf


@torch.no_grad()
def _encode_median_fde(model, loaders, device):
    """Median FDE across neurons, encode task (spikes <- behaviour)."""
    model.eval()
    fdes = []
    for s, sd in loaders.items():
        yt_l, yp_l = [], []
        for b in sd:
            m = b.mask.to(device)
            nt = b.mask.clone().to(device)            # encode: all neural masked
            bt = torch.zeros_like(nt)
            rate, _, _ = model(s, b.spike.to(device), b.behavior.to(device), m, nt, bt)
            mk = b.mask.numpy().astype(bool)
            yt_l.append(b.spike.numpy()[mk]); yp_l.append(rate.cpu().numpy()[mk])
        if yt_l:
            fdes.append(poisson_fde_population(np.vstack(yt_l), np.vstack(yp_l)))
    return float(np.nanmedian(np.hstack(fdes))) if fdes else np.nan


def _cpu_state(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def _pareto_update(cand, err, fde, model):
    """Maintain a Pareto front over (err down, fde up). Store a CPU checkpoint only
    when the new (err, fde) is non-dominated; drop members it dominates. Returns
    (cand, improved)."""
    dominated = any((o['err'] <= err and o['fde'] >= fde) and
                    (o['err'] < err or o['fde'] > fde) for o in cand)
    if dominated:
        return cand, False
    cand = [o for o in cand if not ((err <= o['err'] and fde >= o['fde']) and
                                     (err < o['err'] or fde > o['fde']))]
    cand.append(dict(err=err, fde=fde, w=_cpu_state(model)))
    return cand, True


def train_joint(model, dataloaders, device, n_epochs=300, lr=3e-3, weight_decay=1e-2,
                lam=1.0, schemes=SCHEMES, scheme_p=None, select='joint', fde_weight=0.5,
                err_ref=1.0, fde_ref=0.01, early_stopping_patience=40, verbose=True):
    """select='joint': pick the Pareto-front checkpoint maximising
    fde_weight*(FDE/fde_ref) - (1-fde_weight)*(err/err_ref). Dividing by reference
    magnitudes (err_ref m, fde_ref) puts the two on a common raw scale: with
    err_ref=1.0 and fde_ref=0.01 a 0.01 FDE gain counts as a 1 m error gain, i.e.
    FDE is effectively up-weighted ~100x (its raw scale is ~100x smaller). 'decode':
    legacy min-decode-error. Returns (model, best, hist) with best a dict of the
    selected epoch's err / fde / joint score."""
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    pois_fn = nn.PoissonNLLLoss(log_input=False, reduction='none').to(device)
    hist = {'val_err_m': [], 'val_fde': []}
    rng = random.Random(SEED)
    best_err, best_w, no_improve = np.inf, None, 0          # select='decode'
    cand = []                                               # select='joint' Pareto front
    for ep in tqdm.tqdm(range(n_epochs), desc='joint'):
        model.train()
        for s, b in round_robin_session_batches(dataloaders['train']):
            scheme = rng.choices(schemes, weights=scheme_p)[0]
            m = b.mask.to(device)
            nt, bt = sample_mask(b.mask, scheme)
            nt, bt = nt.to(device), bt.to(device)
            rate, beh, _ = model(s, b.spike.to(device), b.behavior.to(device), m, nt, bt)
            loss = joint_loss(rate, beh, b.spike.to(device), b.behavior.to(device),
                              m, nt, bt, lam, pois_fn)
            opt.zero_grad(); loss.backward(); opt.step()
        err = _decode_median_err_m(model, dataloaders['valid'], meta_info_dict, device)
        fde = _encode_median_fde(model, dataloaders['valid'], device)
        hist['val_err_m'].append(err); hist['val_fde'].append(fde)
        if select == 'decode':
            improved = err < best_err
            if improved:
                best_err, best_w = err, copy.deepcopy(model.state_dict())
        else:
            cand, improved = _pareto_update(cand, err, fde, model)
        no_improve = 0 if improved else no_improve + 1
        if early_stopping_patience and no_improve >= early_stopping_patience:
            if verbose: print(f'early stop @ epoch {ep+1}')
            break
    if select == 'joint':
        errs = np.array([o['err'] for o in cand], float)
        fdes = np.array([o['fde'] for o in cand], float)
        # reference-scale normalisation (common raw scale); fde_ref ~100x smaller
        # than err_ref => FDE up-weighted ~100x. higher = better.
        joint = fde_weight * (fdes / fde_ref) - (1.0 - fde_weight) * (errs / err_ref)
        bi = int(np.nanargmax(joint))
        model.load_state_dict(cand[bi]['w'])
        best = dict(err=float(errs[bi]), fde=float(fdes[bi]), joint=float(joint[bi]))
    else:
        if best_w is not None:
            model.load_state_dict(best_w)
        best = dict(err=float(best_err), fde=np.nan, joint=np.nan)
    return model, best, hist

## 6 · Training

Hyperparameters first, then **tune `lam`** (§6.1), then **sweep depth** (§6.2) at the
tuned `lam`. Reduce `N_EPOCHS` for a quick pass; bump it for the real run.

In [ ]:
N_EPOCHS = 1000
LAYER_DEPTHS = [1, 2, 4, 6, 8, 10, 12]
LATENT_DIM = 64
LAM = 1.0          # behaviour-loss weight; default, overwritten by the §6.1 sweep
# selection reference scales (typical decode err / FDE) -> FDE up-weighted ~23x
ERR_REF, FDE_REF = 0.8, 0.035

### 6.1 · Tuning `lam` (behaviour-loss weight)

`lam` weights the behaviour (MSE) loss against the spike (Poisson) loss *during
training*; it was previously a fixed default, **not** tuned. Here we sweep it at one
depth and pick the value maximising the equal-weight joint criterion on validation
(same criterion as model selection). Because selection is scale-free, `lam` acts only
on the training gradient balance — too small starves decoding, too large starves
encoding. Set `RUN_LAM_SWEEP=False` to skip and keep `LAM=1.0`.

In [ ]:
RUN_LAM_SWEEP = True
LAM_GRID = [0.1, 0.3, 1.0]
LAM_SWEEP_DEPTH = 2          # tune lam at this depth, then reuse for all depths
LAM_SWEEP_EPOCHS = N_EPOCHS

lam_rows = []
if RUN_LAM_SWEEP:
    for lam in LAM_GRID:
        set_random_state(SEED)
        m = JointMaskedTransformer(unit_num_dict, behavior_dim=3, latent_dim=LATENT_DIM,
                                   num_heads=4, ff_dim=LATENT_DIM, num_layers=LAM_SWEEP_DEPTH,
                                   dropout=0.2, input_transform='sqrt',
                                   session_to_room=session_to_room)
        m, best, _ = train_joint(m, dataloaders, device, n_epochs=LAM_SWEEP_EPOCHS,
                                 lam=lam, select='joint', fde_weight=0.5,
                                 err_ref=ERR_REF, fde_ref=FDE_REF, verbose=False)
        lam_rows.append(dict(lam=lam, err_m=best['err'], fde=best['fde'], joint=best['joint']))
        print(f'lam={lam:>5}: decode err={best["err"]:.3f} m  FDE={best["fde"]:.3f}  joint={best["joint"]:.3f}')
    lam_df = pd.DataFrame(lam_rows)
    LAM = float(lam_df.loc[lam_df['joint'].idxmax(), 'lam'])
    print(f'\nselected LAM = {LAM} (max validation joint)')

In [ ]:
if RUN_LAM_SWEEP:
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(lam_df['lam'], lam_df['err_m'], 'o-', color='tab:red', label='decode err (m)')
    ax.set(xscale='log', xlabel='lam (behaviour-loss weight)', ylabel='decode 3-D err (m)')
    ax.tick_params(axis='y', labelcolor='tab:red')
    ax2 = ax.twinx()
    ax2.plot(lam_df['lam'], lam_df['fde'], 's--', color='tab:blue', label='encode FDE')
    ax2.set_ylabel('encode median FDE', color='tab:blue'); ax2.tick_params(axis='y', labelcolor='tab:blue')
    ax.axvline(LAM, color='k', ls=':', lw=1); ax.set_title(f'lam sweep (depth={LAM_SWEEP_DEPTH}); selected={LAM}')
    plt.tight_layout(); plt.show()

### 6.2 · Depth sweep + validation curves

Train one joint model per depth (`LAYER_DEPTHS`) at the tuned `LAM`, selecting each on
the equal-weight joint criterion, and plot validation curves vs epoch to find the plateau.

In [ ]:
joint_models, joint_hist, joint_best = {}, {}, {}
for nl in LAYER_DEPTHS:
    set_random_state(SEED)
    model = JointMaskedTransformer(unit_num_dict, behavior_dim=3,
                                   latent_dim=LATENT_DIM, num_heads=4, ff_dim=LATENT_DIM,
                                   num_layers=nl, dropout=0.2, input_transform='sqrt',
                                   session_to_room=session_to_room)
    model, best, hist = train_joint(model, dataloaders, device, n_epochs=N_EPOCHS,
                                    lam=LAM, select='joint', fde_weight=0.5,
                                    err_ref=ERR_REF, fde_ref=FDE_REF)
    joint_models[nl], joint_hist[nl], joint_best[nl] = model, hist, best
    print(f'num_layers={nl}: selected epoch  decode err={best["err"]:.3f} m  '
          f'FDE={best["fde"]:.3f}  joint={best["joint"]:.3f}')

In [ ]:
# Validation curves vs epoch (find the plateau). decode err, FDE, and the joint
# selection score plotted separately; the dot marks each depth's selected epoch
# (argmax joint over the run = the Pareto-front member that wins, since a weighted-sum
# maximiser is always non-dominated).
def per_epoch_joint(h):
    e = np.array(h['val_err_m'], float); f = np.array(h['val_fde'], float)
    return 0.5 * (f / FDE_REF) - 0.5 * (e / ERR_REF)

depth_cmap = plt.cm.viridis(np.linspace(0, 0.9, len(LAYER_DEPTHS)))
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
for col, nl in zip(depth_cmap, LAYER_DEPTHS):
    h = joint_hist[nl]
    jt = per_epoch_joint(h)
    sel = int(np.nanargmax(jt))
    for ax, y in zip(axes, [h['val_err_m'], h['val_fde'], jt]):
        ax.plot(y, color=col, label=f'{nl}L')
        ax.plot(sel, np.asarray(y)[sel], 'o', color=col, ms=6, mec='k', mew=0.5)
axes[0].set(xlabel='epoch', ylabel='valid 3-D decode error (m)', title='Decoding (↓)')
axes[1].set(xlabel='epoch', ylabel='valid median FDE', title='Encoding (↑)')
axes[2].set(xlabel='epoch', ylabel='joint score', title='Joint selection score (↑)')
axes[0].legend(title='depth', fontsize=7, ncol=2)
plt.tight_layout(); plt.show()

### 6.3 · Metrics vs depth

The selected-epoch validation metrics as a function of transformer depth — decode
error, FDE, and the joint score — to see where added depth stops helping (the joint
panel is what selection optimises). The star marks the best depth.

In [ ]:
depth_df = pd.DataFrame([dict(num_layers=nl, **joint_best[nl]) for nl in LAYER_DEPTHS])
best_depth = int(depth_df.loc[depth_df['joint'].idxmax(), 'num_layers'])
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, (col, label, up) in zip(axes, [('err', 'valid 3-D decode error (m)', False),
                                       ('fde', 'valid median FDE', True),
                                       ('joint', 'joint selection score', True)]):
    ax.plot(depth_df['num_layers'], depth_df[col], 'o-', color='tab:purple')
    yb = depth_df.loc[depth_df['num_layers'] == best_depth, col].iloc[0]
    ax.plot(best_depth, yb, '*', color='gold', ms=18, mec='k', mew=0.6, zorder=5)
    ax.set(xlabel='num layers', ylabel=label, xticks=LAYER_DEPTHS)
fig.suptitle(f'Validation metrics vs depth (selected epoch); best depth = {best_depth}')
plt.tight_layout(); plt.show()
depth_df

## 7 · Encoding evaluation — corr & FDE

Encode task on the held-out test split: spikes ← behaviour. Per-neuron correlation
(`corr`) and Poisson fraction-deviance-explained (`poisson_fde_population`), pooled
over valid test timesteps per session. FDE raw values are kept for stats; the plot
axis is clipped to `[-1, 1]` (FDE is unbounded below).

In [ ]:
@torch.no_grad()
def encoding_records(model, loaders, device):
    model.eval()
    rows = []
    for s, sd in loaders.items():
        yt_l, yp_l = [], []
        for b in sd:
            m = b.mask.to(device)
            nt = b.mask.clone().to(device); bt = torch.zeros_like(nt)
            rate, _, _ = model(s, b.spike.to(device), b.behavior.to(device), m, nt, bt)
            mk = b.mask.numpy().astype(bool)
            yt_l.append(b.spike.numpy()[mk]); yp_l.append(rate.cpu().numpy()[mk])
        if not yt_l:
            continue
        yt, yp = np.vstack(yt_l), np.vstack(yp_l)
        c = corr(yt, yp, axis=0)
        f = poisson_fde_population(yt, yp)
        for j in range(yt.shape[1]):
            rows.append(dict(session=s, bat=parse_bat(s), neuron_idx=j,
                             corr=float(c[j]), fde=float(f[j])))
    return pd.DataFrame(rows)

enc_dfs = {}
for nl in LAYER_DEPTHS:
    df = encoding_records(joint_models[nl], dataloaders['test'], device)
    df['num_layers'] = nl
    enc_dfs[nl] = df
enc_df = pd.concat(enc_dfs.values(), ignore_index=True)
enc_summary = (enc_df.groupby('num_layers')
               .agg(corr_mean=('corr', 'mean'), fde_median=('fde', 'median')))
print(enc_summary)

In [ ]:
# session-level encoding metrics, coloured by bat, with depth comparison
bpal = bat_palette()
enc_sess = (enc_df.groupby(['num_layers', 'bat', 'session'])
            .agg(corr=('corr', 'mean'), fde=('fde', 'median')).reset_index())

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (col, label, clip) in zip(axes, [('corr', 'encoding corr (per session mean)', None),
                                         ('fde', 'encoding FDE (per session median)', (-1, 1))]):
    sns.boxplot(data=enc_sess, x='num_layers', y=col, color='0.85', width=0.5,
                fliersize=0, ax=ax)
    sns.stripplot(data=enc_sess, x='num_layers', y=col, hue='bat', palette=bpal,
                  size=7, jitter=0.15, edgecolor='k', linewidth=0.4, ax=ax, legend=(ax is axes[1]))
    # significance: paired Wilcoxon across sessions between adjacent depths
    piv = enc_sess.pivot_table(index='session', columns='num_layers', values=col)
    for i, (d1, d2) in enumerate(zip(LAYER_DEPTHS[:-1], LAYER_DEPTHS[1:])):
        sub = piv[[d1, d2]].dropna()
        if len(sub) > 1:
            _, p = wilcoxon_safe(sub[d2].values - sub[d1].values)
            add_significance_bracket(ax, i, i + 1, p)
    ax.set(xlabel='num layers', ylabel=label)
    if clip: ax.set_ylim(*clip)
axes[1].legend(title='bat', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=7)
plt.tight_layout(); plt.show()

## 8 · Decoding evaluation — position error & Fréchet (headline)

Decode task on held-out test: position ← spikes. Predictions are de-z-scored to
metres with `behavior_mean`/`behavior_std`. Per-timestep 2-D/3-D Euclidean error and
per-flight discrete Fréchet distance, grouped by `flight_path_index`.

In [ ]:
@torch.no_grad()
def decoding_records(model, loaders, meta, device):
    model.eval()
    rows = []
    for s, sd in loaders.items():
        bm = np.asarray(meta['behavior_mean'][s]).squeeze()[:3]
        bs = np.asarray(meta['behavior_std'][s]).squeeze()[:3]
        for b in sd:
            m = b.mask.to(device)
            bt = b.mask.clone().to(device); nt = torch.zeros_like(bt)
            _, beh_pred, _ = model(s, b.spike.to(device), b.behavior.to(device), m, nt, bt)
            yp_z = beh_pred.cpu().numpy(); yt_z = b.behavior[:, :, :3].numpy()
            mk = b.mask.numpy().astype(bool)
            yt = yt_z * bs[None, None, :] + bm[None, None, :]
            yp = yp_z * bs[None, None, :] + bm[None, None, :]
            B, T = mk.shape
            fpi = np.broadcast_to(b.flight_path_index.numpy()[:, None], (B, T))
            tix = np.broadcast_to(np.arange(T)[None, :], (B, T))
            ytf, ypf = yt[mk], yp[mk]
            err2 = np.sqrt(((ytf[:, :2] - ypf[:, :2]) ** 2).sum(1))
            err3 = np.sqrt(((ytf - ypf) ** 2).sum(1))
            fpf, tif = fpi[mk], tix[mk]
            for i in range(len(err2)):
                rows.append(dict(session=s, bat=parse_bat(s),
                                 flight_id=int(fpf[i]), t_idx=int(tif[i]),
                                 x_true=ytf[i, 0], y_true=ytf[i, 1], z_true=ytf[i, 2],
                                 x_pred=ypf[i, 0], y_pred=ypf[i, 1], z_pred=ypf[i, 2],
                                 err_2d=float(err2[i]), err_3d=float(err3[i])))
    return pd.DataFrame(rows)


def frechet_per_flight(df):
    rows = []
    for (bat, s, fid), g in df.groupby(['bat', 'session', 'flight_id'], sort=False):
        g = g.sort_values('t_idx')
        truth = g[['x_true', 'y_true', 'z_true']].to_numpy(float)
        pred = g[['x_pred', 'y_pred', 'z_pred']].to_numpy(float)
        rows.append(dict(bat=bat, session=s, flight_id=int(fid), n_t=len(g),
                         frechet_2d=discrete_frechet(truth[:, :2], pred[:, :2]),
                         frechet_3d=discrete_frechet(truth, pred)))
    return pd.DataFrame(rows)

dec_ts, dec_fl = {}, {}
for nl in LAYER_DEPTHS:
    ts = decoding_records(joint_models[nl], dataloaders['test'], meta_info_dict, device)
    fl = frechet_per_flight(ts)
    ts['num_layers'] = nl; fl['num_layers'] = nl
    dec_ts[nl], dec_fl[nl] = ts, fl
dec_ts_df = pd.concat(dec_ts.values(), ignore_index=True)
dec_fl_df = pd.concat(dec_fl.values(), ignore_index=True)
print(dec_ts_df.groupby('num_layers')[['err_2d', 'err_3d']].median())
print(dec_fl_df.groupby('num_layers')[['frechet_2d', 'frechet_3d']].median())

In [ ]:
# per-session medians, coloured by bat, with paired-Wilcoxon brackets across depth
metrics = [('err_2d', dec_ts_df, '2-D error (m)'), ('err_3d', dec_ts_df, '3-D error (m)'),
           ('frechet_2d', dec_fl_df, '2-D Fréchet (m)'), ('frechet_3d', dec_fl_df, '3-D Fréchet (m)')]
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
for ax, (col, src, label) in zip(axes, metrics):
    med = src.groupby(['num_layers', 'bat', 'session'])[col].median().reset_index()
    sns.boxplot(data=med, x='num_layers', y=col, color='0.85', width=0.5, fliersize=0, ax=ax)
    sns.stripplot(data=med, x='num_layers', y=col, hue='bat', palette=bpal, size=7,
                  jitter=0.15, edgecolor='k', linewidth=0.4, ax=ax, legend=(ax is axes[-1]))
    piv = med.pivot_table(index='session', columns='num_layers', values=col)
    for i, (d1, d2) in enumerate(zip(LAYER_DEPTHS[:-1], LAYER_DEPTHS[1:])):
        sub = piv[[d1, d2]].dropna()
        if len(sub) > 1:
            _, p = wilcoxon_safe(sub[d2].values - sub[d1].values)
            add_significance_bracket(ax, i, i + 1, p)
    ax.set(xlabel='num layers', ylabel=label)
axes[-1].legend(title='bat', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=7)
plt.tight_layout(); plt.show()

In [ ]:
# example decoded flights at the joint-selected best depth (from §6.3)
ex = dec_ts[best_depth]
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, s in zip(axes, list(ex['session'].unique())[:3]):
    g = ex[ex['session'] == s]
    for fid, gf in list(g.groupby('flight_id'))[:6]:
        gf = gf.sort_values('t_idx')
        ax.plot(gf['x_true'], gf['y_true'], color='k', lw=1.2, alpha=0.7)
        ax.plot(gf['x_pred'], gf['y_pred'], color='tab:red', lw=1.0, alpha=0.7)
    ax.set(title=f'{parse_bat(s)} · {s.split("_")[-1]}', xlabel='x (m)', ylabel='y (m)')
    ax.set_aspect('equal', 'box')
axes[0].plot([], [], 'k', label='true'); axes[0].plot([], [], 'tab:red', label='decoded')
axes[0].legend()
fig.suptitle(f'Decoded flight paths (num_layers={best_depth})'); plt.tight_layout(); plt.show()

## 9 · Interrogating the shared latent

Two free by-products of a single masked model:

1. **Per-neuron embeddings** — rows of the per-session spike read-in weight `(D, K)`
   transposed; each neuron is a `D`-vector. Project all neurons across sessions to 2-D
   and colour by anatomy (bat / probe / depth) to check whether the model recovers
   structure it was never told about.
2. **Cross-modal attention** — fraction of attention mass a neural query token places
   on behaviour keys, per depth: how strongly the latent fuses spikes and behaviour.

In [ ]:
from sklearn.decomposition import PCA
try:
    from umap import UMAP
    _have_umap = True
except Exception:
    _have_umap = False

def neuron_embeddings(model):
    embs, meta_rows = [], []
    for s, lin in model.spike_readin.items():
        W = lin.weight.detach().cpu().numpy().T          # (K, D): row j = neuron j
        embs.append(W)
        for j in range(W.shape[0]):
            meta_rows.append((s, parse_bat(s), j))
    E = np.vstack(embs)
    md_ = pd.DataFrame(meta_rows, columns=['session', 'bat', 'neuron_idx'])
    md_ = md_.merge(neuron_meta, on=['session', 'bat', 'neuron_idx'], how='left')
    return E, md_

E, emb_meta = neuron_embeddings(joint_models[best_depth])
E2 = (UMAP(n_neighbors=30, min_dist=0.3, random_state=SEED).fit_transform(E)
      if _have_umap else PCA(n_components=2, random_state=SEED).fit_transform(E))
emb_meta['e0'], emb_meta['e1'] = E2[:, 0], E2[:, 1]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
sns.scatterplot(data=emb_meta, x='e0', y='e1', hue='bat', palette=bpal, s=12, ax=axes[0], legend=False)
axes[0].set_title('by bat')
sns.scatterplot(data=emb_meta, x='e0', y='e1', hue='probe_id', palette='tab10', s=12, ax=axes[1])
axes[1].set_title('by probe'); axes[1].legend(fontsize=6, ncol=2)
sc = axes[2].scatter(emb_meta['e0'], emb_meta['e1'], c=emb_meta['pos_depth'], cmap='magma', s=12)
axes[2].set_title('by depth'); plt.colorbar(sc, ax=axes[2])
proj = 'UMAP' if _have_umap else 'PCA'
for ax in axes: ax.set(xlabel=f'{proj} 1', ylabel=f'{proj} 2')
fig.suptitle(f'Per-neuron embeddings (read-in weights, num_layers={best_depth})')
plt.tight_layout(); plt.show()

In [ ]:
# Cross-modal attention mass: among a neural query token's attention, what
# fraction lands on behaviour keys vs neural keys? The packaged attention's
# weight-returning path multiplies 0 x (NaN padded value), which corrupts deep
# stacks with padding, so we trim each batch to its no-padding window first
# (padding is always at the end) and run with everything visible (pure fusion).
@torch.no_grad()
def cross_modal_fraction(model, loaders, device, max_batches=8):
    model.eval()
    fracs, seen = [], 0
    for s, sd in loaders.items():
        for b in sd:
            L = int(b.mask.sum(1).min().item())                  # common valid prefix
            if L < 2:
                continue
            spk = b.spike[:, :L].to(device); beh = b.behavior[:, :L].to(device)
            vm = torch.ones(spk.shape[0], L, device=device)
            zt = torch.zeros_like(vm)
            _, _, _, attn = model(s, spk, beh, vm, zt, zt, return_attn_weights=True)
            A = attn[-1].mean(1)                                  # (B, 2L, 2L), head-avg, last layer
            cross = A[:, :L, L:].sum(-1)                          # neural query -> behaviour keys
            within = A[:, :L, :L].sum(-1)                         # neural query -> neural keys
            frac = cross / (cross + within).clamp(min=1e-8)
            fracs.append(float(np.nanmean(frac.cpu().numpy())))
            seen += 1
            if seen >= max_batches: break
        if seen >= max_batches: break
    return float(np.nanmean(fracs)) if fracs else np.nan

for nl in LAYER_DEPTHS:
    cf = cross_modal_fraction(joint_models[nl], dataloaders['test'], device)
    print(f'num_layers={nl}: neural-query attention mass on behaviour = {cf:.3f}')

## 10 · Comparison to the standalone single-task transformers

Does *one* masked multi-task model match the **dedicated** transformers used in the
other notebooks — `Bat_Transformer` (encoder, `preprocess-data-v0j-share.ipynb`) and
`Bat_Transformer_Decoder` (decoder, `decode-position-comparison.ipynb`)? We train both
on the **same splits** at **matched capacity** (`latent_dim`, depth = the joint model's
best depth `best_depth`), with their canonical objectives (encoder: Poisson NLL, FDE
selection; decoder: MSE, neg-Euclidean selection), then evaluate with the *same* metric
functions and overlay joint vs single-task.

In [ ]:
from digitalbat.models.former import Bat_Transformer
from digitalbat.models.former_defaults import make_model_config
from digitalbat.models.decoder import Bat_Transformer_Decoder, bat_decoder_trainer
from digitalbat.training.trainer import bat_former_trainer

BASELINE_DEPTH = best_depth   # match the joint model's selected depth (§6.3)

# ---- standalone ENCODER: Bat_Transformer (behaviour -> spikes) ----
set_random_state(SEED)
enc_cfg = make_model_config(behavior_dim, unit_num_dict, meta_info_dict,
                            session_to_room=session_to_room)
enc_cfg.update(latent_dim=LATENT_DIM, ff_dim=LATENT_DIM, num_heads=4,
               num_layers=BASELINE_DEPTH, dropout_prob=0.2)
encoder_only = Bat_Transformer(**enc_cfg)
bat_former_trainer(encoder_only, dataloaders, device, n_epochs=N_EPOCHS,
                   lr=1e-2, weight_decay=1e-2,
                   loss_fn=nn.PoissonNLLLoss(log_input=False),
                   valid_fn=poisson_fde_population, valid_stat='median',
                   early_stopping_patience=40, save_model=False)

# ---- standalone DECODER: Bat_Transformer_Decoder (spikes -> position) ----
def _neg_euclid_z(y_true, y_pred):
    return -np.sqrt(((y_true - y_pred) ** 2).sum(axis=1))

set_random_state(SEED)
decoder_only = Bat_Transformer_Decoder(input_dim_dict=unit_num_dict, output_dim=3,
                                       latent_dim=LATENT_DIM, num_heads=4, ff_dim=LATENT_DIM,
                                       num_layers=BASELINE_DEPTH, dropout_prob=0.2,
                                       input_transform='sqrt')
bat_decoder_trainer(decoder_only, dataloaders, device, pos_dims=3, n_epochs=N_EPOCHS,
                    lr=3e-3, weight_decay=1e-2, loss_fn=nn.MSELoss(),
                    valid_fn=_neg_euclid_z, valid_stat='median',
                    early_stopping_patience=40, save_model=False)
print('baselines trained at depth', BASELINE_DEPTH)

In [ ]:
# metric functions for the standalone models (their forwards differ from the joint one)
@torch.no_grad()
def encoding_records_former(former, loaders, device):
    former.eval(); rows = []
    for s, sd in loaders.items():
        yt_l, yp_l = [], []
        for b in sd:
            yp = former(s, b.behavior.to(device), b.mask.to(device))
            mk = b.mask.numpy().astype(bool)
            yt_l.append(b.spike.numpy()[mk]); yp_l.append(yp.cpu().numpy()[mk])
        if not yt_l:
            continue
        yt, yp = np.vstack(yt_l), np.vstack(yp_l)
        c = corr(yt, yp, axis=0); f = poisson_fde_population(yt, yp)
        for j in range(yt.shape[1]):
            rows.append(dict(session=s, bat=parse_bat(s), neuron_idx=j,
                             corr=float(c[j]), fde=float(f[j])))
    return pd.DataFrame(rows)

@torch.no_grad()
def decoding_records_decoder(dec, loaders, meta, device):
    dec.eval(); rows = []
    for s, sd in loaders.items():
        bm = np.asarray(meta['behavior_mean'][s]).squeeze()[:3]
        bs = np.asarray(meta['behavior_std'][s]).squeeze()[:3]
        for b in sd:
            yp_z = dec(s, b.spike.to(device), b.mask.to(device)).cpu().numpy()
            yt_z = b.behavior[:, :, :3].numpy(); mk = b.mask.numpy().astype(bool)
            yt = yt_z * bs[None, None, :] + bm[None, None, :]
            yp = yp_z * bs[None, None, :] + bm[None, None, :]
            B, T = mk.shape
            fpi = np.broadcast_to(b.flight_path_index.numpy()[:, None], (B, T))
            tix = np.broadcast_to(np.arange(T)[None, :], (B, T))
            ytf, ypf, fpf, tif = yt[mk], yp[mk], fpi[mk], tix[mk]
            e2 = np.sqrt(((ytf[:, :2] - ypf[:, :2]) ** 2).sum(1))
            e3 = np.sqrt(((ytf - ypf) ** 2).sum(1))
            for i in range(len(e2)):
                rows.append(dict(session=s, bat=parse_bat(s), flight_id=int(fpf[i]), t_idx=int(tif[i]),
                                 x_true=ytf[i, 0], y_true=ytf[i, 1], z_true=ytf[i, 2],
                                 x_pred=ypf[i, 0], y_pred=ypf[i, 1], z_pred=ypf[i, 2],
                                 err_2d=float(e2[i]), err_3d=float(e3[i])))
    return pd.DataFrame(rows)

# joint (at matched depth) vs single-task
enc_cmp = pd.concat([
    enc_dfs[BASELINE_DEPTH].assign(model='joint'),
    encoding_records_former(encoder_only, dataloaders['test'], device).assign(model='encoder-only'),
], ignore_index=True)
dec_base_ts = decoding_records_decoder(decoder_only, dataloaders['test'], meta_info_dict, device)
dec_cmp_ts = pd.concat([dec_ts[BASELINE_DEPTH].assign(model='joint'),
                        dec_base_ts.assign(model='decoder-only')], ignore_index=True)
fre_cmp = pd.concat([dec_fl[BASELINE_DEPTH].assign(model='joint'),
                     frechet_per_flight(dec_base_ts).assign(model='decoder-only')], ignore_index=True)
print('encoding:', enc_cmp.groupby('model').agg(corr=('corr','mean'), fde=('fde','median')).to_dict())
print('decoding:', dec_cmp_ts.groupby('model')[['err_2d','err_3d']].median().to_dict())

In [ ]:
def _compare_box(ax, med, col, label, order, clip=None):
    sns.boxplot(data=med, x='model', y=col, order=order, color='0.85', width=0.5,
                fliersize=0, ax=ax)
    sns.stripplot(data=med, x='model', y=col, order=order, hue='bat', palette=bpal,
                  size=7, jitter=0.15, edgecolor='k', linewidth=0.4, ax=ax, legend=False)
    piv = med.pivot_table(index='session', columns='model', values=col)
    if set(order) <= set(piv.columns):
        sub = piv[order].dropna()
        if len(sub) > 1:
            _, p = wilcoxon_safe(sub[order[1]].values - sub[order[0]].values)
            add_significance_bracket(ax, 0, 1, p)
    ax.set(xlabel='', ylabel=label); ax.set_xticklabels(ax.get_xticklabels(), rotation=15)
    if clip: ax.set_ylim(*clip)

# encoding: joint vs encoder-only
enc_cmp_sess = (enc_cmp.groupby(['model', 'bat', 'session'])
                .agg(corr=('corr', 'mean'), fde=('fde', 'median')).reset_index())
fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
_compare_box(axes[0], enc_cmp_sess, 'corr', f'encoding corr (depth {BASELINE_DEPTH})', ['encoder-only', 'joint'])
_compare_box(axes[1], enc_cmp_sess, 'fde', 'encoding FDE', ['encoder-only', 'joint'], clip=(-1, 1))
fig.suptitle('Encoding: joint vs standalone Bat_Transformer'); plt.tight_layout(); plt.show()

# decoding: joint vs decoder-only
specs = [('err_2d', dec_cmp_ts, '2-D err (m)'), ('err_3d', dec_cmp_ts, '3-D err (m)'),
         ('frechet_2d', fre_cmp, '2-D Fréchet (m)'), ('frechet_3d', fre_cmp, '3-D Fréchet (m)')]
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
for ax, (col, src, label) in zip(axes, specs):
    med = src.groupby(['model', 'bat', 'session'])[col].median().reset_index()
    _compare_box(ax, med, col, label, ['decoder-only', 'joint'])
fig.suptitle('Decoding: joint vs standalone Bat_Transformer_Decoder'); plt.tight_layout(); plt.show()

## 11 · Notes & next steps

* **Model selection** uses the joint criterion (§5): the Pareto front over
  `(−decode-error, +FDE)`, scored by reference-scaled
  `0.5·(FDE/FDE_REF) − 0.5·(err/ERR_REF)` with `ERR_REF=1 m`, `FDE_REF=0.01` — so
  FDE (raw scale ~100× smaller) is up-weighted ~100×. Tune `ERR_REF`/`FDE_REF` (or
  `fde_weight`) to shift the encoding↔decoding trade-off.
* **`lam`** is tuned in §6.1 (it was previously fixed at 1.0); `scheme_p` (per-task
  sampling weights) is the other knob — up-weighting `decode` trades encoding for
  decoding during training.
* §10 compares the joint model to the dedicated single-task `Bat_Transformer` /
  `Bat_Transformer_Decoder` at matched capacity. To also benchmark against the
  Naive-Bayes / GLM-MAP baselines, fold `model='joint'` rows into
  `decode-position-comparison.ipynb`'s `MODEL_ORDER` pipeline (same
  `err_2d`/`err_3d`/`frechet_*` columns).
* This is **regime 1** (held-out flights within seen sessions); cross-session /
  cross-animal generalisation is out of scope here.